# Other plants and burned areas — Coimbra district

**Purpose.** Counts behind the design in `docs/research_questions.md`:
*Acacia* presences vs. other-plant observations (used as classes), and
the effect of the burned-area filter. Year distribution and observer
concentration are already in
`../02_citizen_science/acacia_observations_exploration.ipynb`.

**Scope.** Coimbra district (iNaturalist place 8422).
All iNaturalist observations are treated as the reference year 2023
(adopted 2026-10-21); the disturbance rule is still a proposal.
Downloads are cached in `data/raw/` and `data/external/`, documented in
`data/README.md`; figures go to `results/figures/meeting_2026-10/`
(not versioned).

## 0. Setup

Imports, folder paths and constants used throughout:
- `PLACE_ID = 8422`: the iNaturalist place for the Coimbra district;
- `ACACIA = 47452` and `PLANTAE = 47126`: iNaturalist taxon ids for
  the genus *Acacia* and for all plants;
- `REF_YEAR = 2023`: the year all observations are treated as;
- `METRIC_CRS = 3763`: PT-TM06/ETRS89, a projection in metres, so
  distances and 1 km cells are correct.

The folders for figures and tables are created if they do not exist.

In [ ]:
import json
import time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import requests
from shapely.geometry import shape

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW_CS = ROOT / "data" / "raw" / "citizen_science"
RAW_FIRE = ROOT / "data" / "external" / "auxiliary" / "icnf_burned_areas"
FIG = ROOT / "results" / "figures" / "meeting_2026-10"
TAB = ROOT / "results" / "tables" / "meeting_2026-10"
for d in (RAW_FIRE, FIG, TAB):
    d.mkdir(parents=True, exist_ok=True)

PLACE_ID = 8422          # iNaturalist "Coimbra, PT" (district)
ACACIA = 47452           # genus Acacia
PLANTAE = 47126          # kingdom Plantae
REF_YEAR = 2023          # all observations are treated as this year
METRIC_CRS = 3763        # PT-TM06/ETRS89

## 1. Download: other-plant observations (all years)

**What it does.** Asks the iNaturalist API for every plant observation
in the Coimbra district (`place_id=8422`), excluding *Acacia*, and
saves them as CSV files in `data/raw/citizen_science/`.

**Why.** These observations become the other-plant classes: they were
made by the same people, in the same places, as the *Acacia*
presences, so the model learns *Acacia* vs. other vegetation rather
than "visited vs. not visited" (Phillips et al., 2009).

**How.**
- The download is split into three date ranges (before 2021,
  2021–2025, from 2026), because 2021–2025 was fetched first and the
  other years later. Each range is one file.
- The API returns 200 observations per page. Pages are requested in
  order of id, waiting 1 s between requests (iNaturalist's rate
  limit).
- It is **resumable**: each page is appended to a `.partial.csv` file,
  and if the download stops it restarts from the last saved id. When
  a range finishes, the partial file is renamed.
- If a file already exists, it is not downloaded again.

**Output.** A table with the number of observations per year.

In [ ]:
# Downloaded in date ranges; 2021–2025 was fetched first, the other ranges later.
BG_RANGES = {"before_2021": (None, "2020-12-31"),
             "2021_2025": ("2021-01-01", "2025-12-31"),
             "from_2026": ("2026-01-01", None)}
bg_file = lambda key: RAW_CS / f"inaturalist_plantae_no_acacia_coimbra_{key}.csv"

def date_params(key):
    d1, d2 = BG_RANGES[key]
    return {k: v for k, v in dict(d1=d1, d2=d2).items() if v}

def download_background(key):
    """Resumable: appends each page to a partial file and restarts from the last id saved."""
    partial = bg_file(key).with_suffix(".partial.csv")
    url = "https://api.inaturalist.org/v1/observations"
    last_id = int(pd.read_csv(partial, usecols=["id"])["id"].max()) if partial.exists() else 0
    params = dict(place_id=PLACE_ID, taxon_id=PLANTAE, without_taxon_id=ACACIA, **date_params(key),
                  order_by="id", order="asc", per_page=200, id_above=last_id)
    while True:
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        res = r.json()["results"]
        if not res:
            break
        rows = []
        for o in res:
            if not o.get("geojson"):
                continue
            lon, lat = o["geojson"]["coordinates"]
            t = o.get("taxon") or {}
            rows.append(dict(id=o["id"], observed_on=o.get("observed_on"),
                             taxon_id=t.get("id"), taxon_name=t.get("name"), taxon_rank=t.get("rank"),
                             latitude=lat, longitude=lon,
                             positional_accuracy=o.get("positional_accuracy"),
                             quality_grade=o.get("quality_grade"),
                             user_login=(o.get("user") or {}).get("login"),
                             license_code=o.get("license_code")))
        pd.DataFrame(rows).to_csv(partial, mode="a", header=not partial.exists(), index=False)
        params["id_above"] = res[-1]["id"]
        time.sleep(1)  # iNaturalist asks for <= 60 requests/min
    partial.rename(bg_file(key))

for key in BG_RANGES:
    if not bg_file(key).exists():
        download_background(key)
bg_raw = pd.concat([pd.read_csv(bg_file(key)) for key in BG_RANGES], ignore_index=True)
bg_raw.groupby(pd.to_datetime(bg_raw["observed_on"], errors="coerce").dt.year).size()

### Obscured coordinates

**What it does.** Downloads the ids of observations whose location is
hidden, and stores them in `*_obscured_ids.csv` (one file per range).

**Why.** When an observer or a species has geoprivacy set to
"obscured" or "private", iNaturalist moves the point to a random spot
inside a ~0.2° cell (several km). Those coordinates are useless for
10 m pixels. They are excluded in section 3. (These were the
rectangular clouds of points seen in the first maps.)

**How.** Two queries per range, one for user geoprivacy and one for
taxon geoprivacy, returning ids only.

**Output.** The number of obscured ids.

In [ ]:
obscured_file = lambda key: RAW_CS / f"inaturalist_plantae_no_acacia_coimbra_{key}_obscured_ids.csv"

def download_obscured_ids(key):
    url = "https://api.inaturalist.org/v1/observations"
    ids = set()
    for privacy in [dict(geoprivacy="obscured,private"), dict(taxon_geoprivacy="obscured,private")]:
        params = dict(place_id=PLACE_ID, taxon_id=PLANTAE, without_taxon_id=ACACIA, **date_params(key),
                      order_by="id", order="asc", per_page=200, id_above=0, only_id="true", **privacy)
        while True:
            r = requests.get(url, params=params, timeout=60)
            r.raise_for_status()
            res = r.json()["results"]
            if not res:
                break
            ids.update(o["id"] for o in res)
            params["id_above"] = res[-1]["id"]
            time.sleep(1)
    pd.DataFrame({"id": sorted(ids)}).to_csv(obscured_file(key), index=False)

for key in BG_RANGES:
    if not obscured_file(key).exists():
        download_obscured_ids(key)
obscured_ids = set().union(*(set(pd.read_csv(obscured_file(key))["id"]) for key in BG_RANGES))
len(obscured_ids)

## 2. Download: ICNF burned areas, 1975–2025

**What it does.** Downloads the burned-area polygons published by
ICNF (Instituto da Conservação da Natureza e das Florestas) through
its WFS service, and saves each layer as GeoJSON in
`data/external/auxiliary/icnf_burned_areas/`.

**Why.** For the burned-area filter (section 5): if a place burned
between the observation and 2023, the 2023 image no longer shows what
was observed.

**How.**
- ICNF publishes grouped layers before 2009 (`1975_1989`,
  `1990_1999`, `2000_2008`) and one layer per year from 2009. 1975 is
  the first year available and covers the oldest observation (other
  plants from 1976).
- Every fire has a year (`Ano`). Most fires from 2014 also have a
  start date (`DH_Inicio`); older ones do not.
- Layers cover all of continental Portugal (not clipped).

**Output.** A table with the number of fires per year, and how many
have a start date.

In [ ]:
FIRE_LAYERS = ["1975_1989", "1990_1999", "2000_2008", *map(str, range(2009, 2026))]

def download_fires(layer):
    f = RAW_FIRE / f"ardida_{layer}.geojson"
    if not f.exists():
        url = "https://si.icnf.pt/wfs/areas_ardidas"
        params = dict(service="WFS", version="2.0.0", request="GetFeature",
                      typeNames=f"BDG:ardida_{layer}", outputFormat="application/json")
        r = requests.get(url, params=params, timeout=600)
        r.raise_for_status()
        f.write_bytes(r.content)
    return gpd.read_file(f).set_crs(METRIC_CRS, allow_override=True)

fires = pd.concat([download_fires(layer) for layer in FIRE_LAYERS], ignore_index=True)
fires["fire_date"] = pd.to_datetime(fires.get("DH_Inicio"), utc=True, errors="coerce").dt.tz_localize(None)
fires.groupby("Ano").agg(n=("Ano", "size"), with_date=("fire_date", "count"))

## 3. Load and prepare

**What it does.** Puts the *Acacia* presences and the other plants in
the same format, ready for the checks.

**How.**
- Loads the district boundary from iNaturalist (used to draw the maps
  and select the fires inside the district).
- `to_gdf` turns each table into points with a date and a year, in
  the Portuguese metric projection (EPSG:3763, metres), and drops
  observations without a date.
- Removes obscured coordinates: the *Acacia* file has its own flag
  (`coordinates_obscured`); other plants use the ids from section 1.
- Marks the person with the most *Acacia* observations as
  "Observer A" (anonymised) and everyone else as "Others", in both
  tables. This is used to check whether that person's footprint is
  shared by the other plants.
- `base` is a small helper that draws the district outline on a map.

**Output.** How many observations were dropped as obscured, and how
many remain (with their year range).

In [ ]:
place = requests.get(f"https://api.inaturalist.org/v1/places/{PLACE_ID}", timeout=60).json()["results"][0]
district = gpd.GeoDataFrame(geometry=[shape(place["geometry_geojson"])], crs=4326).to_crs(METRIC_CRS)

def to_gdf(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["observed_on"], errors="coerce")
    df["year"] = df["date"].dt.year
    df = df[df["year"].notna()]
    return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs=4326).to_crs(METRIC_CRS)

acacia_raw = pd.read_csv(RAW_CS / "inaturalist_acacia_coimbra_pilot.csv")
acacia = to_gdf(acacia_raw[~acacia_raw["coordinates_obscured"]])
bg = to_gdf(bg_raw[~bg_raw["id"].isin(obscured_ids)])
print(f"Dropped obscured: Acacia {acacia_raw['coordinates_obscured'].sum()}, "
      f"other plants {bg_raw['id'].isin(obscured_ids).sum()}")
print(f"Dated, non-obscured: Acacia {len(acacia)} ({int(acacia.year.min())}–{int(acacia.year.max())}), "
      f"other plants {len(bg)} ({int(bg.year.min())}–{int(bg.year.max())})")

# The dominant observer is anonymised as "Observer A"
dominant = acacia["user_login"].value_counts().index[0]
for df in (acacia, bg):
    df["observer"] = (df["user_login"] == dominant).map({True: "Observer A", False: "Others"})

def base(ax, title):
    district.boundary.plot(ax=ax, color="black", linewidth=0.8)
    ax.set_title(title); ax.set_axis_off()

## 4. Presences vs. other plants (all years)

**What it does.** Maps *Acacia* presences (red) on top of other-plant
observations (grey), in three panels: all observers, Observer A
only, and everyone else.

**Why.** Using other plants only works if they come from the same
places as the presences. The map shows whether that holds, and how
much of it depends on Observer A.

**Output.** `results/figures/meeting_2026-10/presences_vs_background.png`.

In [ ]:
groups = [("All observers", acacia, bg),
          ("Observer A", acacia[acacia.observer == "Observer A"], bg[bg.observer == "Observer A"]),
          ("Others", acacia[acacia.observer == "Others"], bg[bg.observer == "Others"])]
fig, axes = plt.subplots(1, 3, figsize=(18, 7), sharex=True, sharey=True)
for ax, (label, p, b) in zip(axes, groups):
    b.plot(ax=ax, markersize=0.5, color="#7f7f7f", alpha=0.3, label=f"other plants (n={len(b)})")
    p.plot(ax=ax, markersize=2, color="#c0504d", alpha=0.7, label=f"Acacia (n={len(p)})")
    base(ax, label); ax.legend(loc="lower right", markerscale=5)
plt.tight_layout(); plt.savefig(FIG / "presences_vs_background.png", dpi=200); plt.show()

### Footprint overlap on a 1 km grid

**What it does.** Divides the district into 1 km × 1 km cells and, for
each group (all, Observer A, others), counts:
- the cells with *Acacia* and the cells with other plants;
- `pct_acacia_cells_with_background`: of the cells with *Acacia*, the
  share that also have other plants. High = the other plants cover
  the places where *Acacia* was seen;
- `pct_background_cells_with_acacia`: of the cells with other plants,
  the share that also have *Acacia*. Low = other plants cover much
  more territory than *Acacia*;
- `ratio_bg_per_presence`: other-plant observations per *Acacia*
  presence.

**Output.** The table, also saved as
`results/tables/meeting_2026-10/presence_background_counts.csv`.

In [ ]:
def cells(df, size=1000):
    return set(zip((df.geometry.x // size).astype(int), (df.geometry.y // size).astype(int)))

rows = []
for label, p, b in groups:
    cp, cb = cells(p), cells(b)
    rows.append(dict(group=label, acacia=len(p), background=len(b),
                     ratio_bg_per_presence=round(len(b) / max(len(p), 1), 1),
                     acacia_cells_1km=len(cp), background_cells_1km=len(cb),
                     pct_acacia_cells_with_background=round(100 * len(cp & cb) / max(len(cp), 1), 1),
                     pct_background_cells_with_acacia=round(100 * len(cp & cb) / max(len(cb), 1), 1)))
counts = pd.DataFrame(rows)
counts.to_csv(TAB / "presence_background_counts.csv", index=False)
counts

## 5. Burned-area filter

**What it does.** Counts how many observations the proposed burned-area
rule (not yet a decision) would remove.

**The rule.** An observation is removed if it lies inside an area that
burned **between the observation and 2023**:
- observation before 2023 → fire after the observation and up to the
  end of 2023;
- observation after 2023 → fire from the start of 2023 up to the
  observation.

**Why.** All observations are treated as 2023. If the place burned in
between, the 2023 pixel shows burned ground or regrowth, not what was
observed. *Acacia* is not gone after a fire (Riveiro et al., 2025) —
the problem is that the pixel no longer matches the label.

**How.** Fires with a start date are compared by date. Fires with only
a year are compared by year; if the fire is in the same year as the
observation the order is unknown, so it is counted as burned
(conservative) and reported in `of_which_same_year`.

**Output.** The table, also saved as
`results/tables/meeting_2026-10/burned_filter_counts.csv`.

In [ ]:
fires_local = fires[fires.intersects(district.geometry.iloc[0])]
ref_start, ref_end = pd.Timestamp(f"{REF_YEAR}-01-01"), pd.Timestamp(f"{REF_YEAR}-12-31")

def burned_between(points):
    j = gpd.sjoin(points[["id", "date", "year", "geometry"]],
                  fires_local[["Ano", "fire_date", "geometry"]], predicate="within")
    dated = j["fire_date"].notna()
    lo = j["date"].where(j["date"] < ref_start, ref_start)
    hi = j["date"].where(j["date"] > ref_end, ref_end)
    by_date = dated & (j["fire_date"] >= lo) & (j["fire_date"] <= hi)
    by_year = ~dated & j["Ano"].between(j["year"].clip(upper=REF_YEAR), j["year"].clip(lower=REF_YEAR))
    ambiguous = ~dated & (j["Ano"] == j["year"])
    return points["id"].isin(j.loc[by_date | by_year, "id"]), points["id"].isin(j.loc[ambiguous, "id"])

acacia["burned"], acacia["burned_same_year"] = burned_between(acacia)
bg["burned"], bg["burned_same_year"] = burned_between(bg)
burn_table = pd.DataFrame([
    dict(set=name, n=len(df), removed=int(df.burned.sum()), pct_removed=round(100 * df.burned.mean(), 1),
         of_which_same_year=int(df.burned_same_year.sum()))
    for name, df in [("Acacia", acacia), ("Other plants", bg)]])
burn_table.to_csv(TAB / "burned_filter_counts.csv", index=False)
print(f"Burned areas intersecting the district, 1975–2025: {len(fires_local)}")
burn_table

### Areas burned in 2025

**What it does.** Counts *Acacia* presences inside areas burned in
2025.

**Why.** The rule keeps them (the 2023 image predates the fire), but
the number is the reason 2023 is the reference year and not a later
one: with 2025 images these places would be burned ground.

In [ ]:
in_2025 = gpd.sjoin(acacia[["id", "geometry"]], fires_local.loc[fires_local["Ano"] == 2025, ["geometry"]], predicate="within")["id"].nunique()
print(f"Acacia in areas burned in 2025: {in_2025} of {len(acacia)} ({100 * in_2025 / len(acacia):.1f}%)")

### Map: presences and burned areas

**What it does.** Draws the burned areas in the district coloured by
fire year, the *Acacia* presences kept (blue) and those removed by the
rule (black ×).

**Output.** `results/figures/meeting_2026-10/burned_areas_overlay.png`.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
fires_local.plot(ax=ax, column="Ano", legend=True, alpha=0.5, cmap="YlOrRd", legend_kwds=dict(label="Fire year", shrink=0.6))
acacia[~acacia.burned].plot(ax=ax, markersize=2, color="#4f81bd", alpha=0.6)
acacia[acacia.burned].plot(ax=ax, markersize=30, color="black", marker="x")
base(ax, f"Acacia and ICNF burned areas 1975–2025 (black ×: removed by the rule, {int(acacia.burned.sum())})")
plt.tight_layout(); plt.savefig(FIG / "burned_areas_overlay.png", dpi=200); plt.show()

### Save the cleaned observations

**What it does.** Writes the *Acacia* presences and other-plant
observations used above — dated, non-obscured, with the
`burned` flag from the rule — to `data/interim/citizen_science/`, so
later notebooks start from the same data without
re-downloading or re-applying the rule. Observations are **not**
removed here; the `burned` column lets each experiment decide.

In [ ]:
INTERIM_CS = ROOT / "data" / "interim" / "citizen_science"
INTERIM_CS.mkdir(parents=True, exist_ok=True)
keep = ["id", "observed_on", "year", "taxon_id", "taxon_name", "taxon_rank", "latitude", "longitude",
        "positional_accuracy", "quality_grade", "observer", "burned", "burned_same_year"]
acacia[keep].to_csv(INTERIM_CS / "inat_acacia_clean.csv", index=False)
bg[keep].to_csv(INTERIM_CS / "inat_other_plants_clean.csv", index=False)
print(f"Saved {len(acacia)} Acacia and {len(bg)} other-plant observations to {INTERIM_CS.relative_to(ROOT)}")

## 6. Expert reference polygons

**What it does.** Loads the *A. dealbata* polygons sent by the
supervisor (the expert reference data for the RQ1 upper bound) and
counts what is available in the Coimbra district.

**Why.** Before using them we need to know how many there are, how
spread out they are (for spatial training/test splits), how they
relate to COS 2023, and whether they cover the same places as the
iNaturalist observations. The labels are independent (collected by the
experts), but shared locations matter for the training/test split.

**How.**
- Two files: `TNE` (Sentinel-2 tile TNE, which covers the district)
  and `notTNE` (rest of continental Portugal). Only `classifica = 1`
  (*A. dealbata* present) is used; the other polygons lie mostly on
  water and built-up areas.
- Many positives are ~5 m-radius circles (about one 10 m pixel),
  likely field points with a buffer; the rest are larger patches.

**Output.** A table per file, and counts for the district.

In [ ]:
EXPERT = ROOT / "data" / "raw" / "expert_reference"
expert = pd.concat([
    gpd.read_file(EXPERT / "TNE_17MAY2026_5.gpkg").assign(source="TNE"),
    gpd.read_file(EXPERT / "notTNE_18MAY2026_6.gpkg").assign(source="notTNE"),
], ignore_index=True)
expert = gpd.GeoDataFrame(expert, crs=METRIC_CRS)
expert["classifica"] = pd.to_numeric(expert["classifica"], errors="coerce")
expert["ha"] = expert.area / 1e4
expert["point_like"] = expert["ha"] < 0.01  # ~5 m-radius circles

print(expert.groupby(["source", "classifica"], dropna=False).size().unstack(fill_value=0))
pos = expert[expert["classifica"] == 1]
pos.groupby("source").agg(polygons=("ha", "size"), area_ha=("ha", "sum"),
                          point_like=("point_like", "sum"), over_1_ha=("ha", lambda h: (h > 1).sum())).round(1)

### In the district: spread, COS 2023 and iNaturalist

**What it does.**
- **Spread:** in how many 1 km and 10 km cells the district polygons
  fall. Few 10 km cells = few independent blocks for spatial
  validation.
- **COS 2023:** how many polygons touch a "Florestas de acácias"
  (class 5.1.1.7) polygon. Reading COS from the zip takes a minute or
  two.
- **iNaturalist:** how many *Acacia* presences lie inside or near
  (50 m, 100 m) a polygon, and how many of those are Observer A's. A
  strong overlap means experts and citizen scientists visited the
  same places: testing the citizen-science model on these polygons
  would evaluate it where it was trained.

**Output.** The table, also saved as
`results/tables/meeting_2026-10/expert_reference_counts.csv`.

In [ ]:
pos_d = pos[pos.intersects(district.geometry.iloc[0])].copy()
centroids = pos_d.geometry.centroid

cos = gpd.read_file(ROOT / "data" / "raw" / "satellite" / "cos" / "COS2023v1-S2-gpkg.zip",
                    layer="COS2023v1", where="COS23_n4_C = '5.1.1.7'",
                    bbox=tuple(pos_d.total_bounds)).to_crs(METRIC_CRS)
touch_cos = gpd.sjoin(pos_d[["geometry"]], cos[["geometry"]], predicate="intersects").index.unique()

acacia_d = acacia[acacia.within(district.geometry.iloc[0])]
rows = [dict(measure="polygons in district", value=len(pos_d)),
        dict(measure="area in district (ha)", value=round(pos_d["ha"].sum(), 1)),
        dict(measure="distinct 1 km cells", value=len(cells(centroids.to_frame("geometry").set_geometry("geometry")))),
        dict(measure="distinct 10 km cells", value=len(cells(centroids.to_frame("geometry").set_geometry("geometry"), 10000))),
        dict(measure="touching COS 'Florestas de acácias'", value=len(touch_cos))]
for buf in (0, 50, 100):
    near = gpd.sjoin(acacia_d[["id", "observer", "geometry"]],
                     pos_d.assign(geometry=pos_d.buffer(buf))[["geometry"]], predicate="within").drop_duplicates("id")
    rows.append(dict(measure=f"iNat presences within {buf} m", value=len(near)))
    rows.append(dict(measure=f"  of which Observer A (%)", value=round(100 * (near.observer == "Observer A").mean(), 1)))
expert_counts = pd.DataFrame(rows)
expert_counts.to_csv(TAB / "expert_reference_counts.csv", index=False)
print(f"iNat presences in the district: {len(acacia_d)}")
expert_counts

## Summary

All years, all observations treated as 2023; obscured coordinates
excluded (43 *Acacia*, 9,903 other plants).

- **Counts:** 7,898 dated *Acacia* presences (1992–2026) vs. 110,576
  other-plant observations (1976–2026), 14 per presence.
- **Observer A also records other plants:** 36,080 records (33% of
  other plants). 96.7% of Observer A's *Acacia* 1 km cells also hold
  their other-plant records; for other observers, 94.2%.
- **Footprints are not identical:** only 26.6% of other-plant 1 km
  cells hold *Acacia*.
- **Burned-area rule (fire between observation and 2023, ICNF
  1975–2025):** removes 35 *Acacia* (0.4%) and 2,880 other plants
  (2.6%). Other plants lose more because they include older records
  and the 2017 fires (≈268,000 ha in the district) burned after many
  of them. Same-year cases without a fire date: 0 *Acacia*, 3 other
  plants (counted as burned).
- **2,299 *Acacia* presences (29.1%) lie in areas burned in 2025**
  (≈70,600 ha in the district). Kept under the rule, since the 2023
  image predates the fire.
- **Expert reference polygons (*A. dealbata*):** 1,849 in the
  district (244 ha), all in tile TNE, in only 15 cells of 10 km.
  1,084 touch a COS 2023 "Florestas de acácias" polygon. 582
  iNaturalist presences lie within 50 m of a polygon, 96.2% of them
  Observer A's (all 70 inside a polygon are). Labels are independent
  (collected by the experts), but the locations overlap, so training
  and test must be split by spatial blocks.
